# Metric Presentation and Visualization
## Necessary packages and functions call

- DDPM-TS: Interpretable Diffusion for Time Series Generation
- Metrics: 
    - discriminative_metrics
    - predictive_metrics
    - visualization

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

from Utils.metric_utils import display_scores
from Utils.discriminative_metric import discriminative_score_metrics
from Utils.predictive_metric import predictive_score_metrics

## Data Loading

Load original dataset and preprocess the loaded data.

In [ ]:
iterations = 5
dataset_name = 'ETTh1'
seq_length = 169
# ori_data = np.load('../toy_exp/samples/energy_ground_truth_24_train.npy')
# ori_data = np.load(f'../energy_results/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
ori_data = np.load(f'../energy_results/ETTh1/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')
fake_data = np.load('../energy_results/ETTh1/ddpm_fake_energy_0_to_1.npy')

## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [3]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('energy:')
display_scores(discriminative_score)
print()

Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Please use tf.global_variables instead.


training: 100%|██████████| 2000/2000 [00:47<00:00, 41.73it/s]


Iter 0:  0.10160519125683065 , 0.6584699453551912 , 0.54474043715847 



training: 100%|██████████| 2000/2000 [00:47<00:00, 42.30it/s]


Iter 1:  0.0744535519125683 , 0.6444672131147541 , 0.5044398907103825 



...

Iter 2:  0.04952185792349728 , 0.6007513661202186 , 0.49829234972677594 



training: 100%|██████████| 2000/2000 [00:46<00:00, 42.75it/s]


Iter 3:  0.225922131147541 , 0.7756147540983607 , 0.6762295081967213 



training: 100%|██████████| 2000/2000 [00:46<00:00, 42.94it/s]


Iter 4:  0.26963797814207646 , 0.7708333333333334 , 0.7684426229508197 

energy:
Final Score:  0.14422814207650275 ± 0.12111161687637721



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [4]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('energy:')
display_scores(predictive_score)
print()

training: 100%|██████████| 5000/5000 [01:23<00:00, 59.80it/s]


0  epoch:  0.05335225923416827 



training: 100%|██████████| 5000/5000 [01:23<00:00, 59.69it/s]


1  epoch:  0.0518406350698679 



training: 100%|██████████| 5000/5000 [01:23<00:00, 59.71it/s]


2  epoch:  0.05153097391581806 



training: 100%|██████████| 5000/5000 [01:24<00:00, 59.22it/s]


3  epoch:  0.0519786300056385 



training: 100%|██████████| 5000/5000 [01:22<00:00, 60.41it/s]


4  epoch:  0.053366964072497414 

energy:
Final Score:  0.05241389245959803 ± 0.0010907048765558304

